<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">04 Mobile Algorithms</h1>
<hr>


## **3. Tit-for-Tat Spectrum Management**

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>



### 3.1 Introduction 

Distributed systems often require multiple autonomous agents to make decisions without any central coordination. In wireless networks, this happens naturally when many devices attempt to use the same spectrum at the same time. Each device wants to transmit its data, but transmitting at the wrong moment can cause interference and reduce the performance of the entire network.

In this notebook, we examine how concepts from game theory - especially the Prisoner’s Dilemma - and simple behavioral strategies such as Tit-for-Tat can help us understand and design mechanisms like backoff algorithms in medium access control (MAC) protocols.


By simulating repeated interactions between devices, we will observe how cooperation, defection, and adaptive strategies affect their achievable bandwidth.


### 3.2 Game Theory: The Prisoner’s Dilemma

*Definition:
The Prisoner’s Dilemma is a classic scenario in game theory that shows how rational individuals, acting in their own self-interest, can end up with worse outcomes than if they had cooperated.*

The Prisoner’s Dilemma (PD) is a two-player game in which each player has two possible actions:
**Cooperate (C)** or **Defect (D)**. The resulting payoffs follow a characteristic pattern:

* If both players cooperate, both receive a good outcome.
* If both defect, both receive a poor outcome.
* If one defects while the other cooperates, the defector gains a large benefit while the cooperator suffers the worst outcome.

This structure models situations in which a shared resource must be used responsibly. In wireless networks, for example, if both devices “grab the channel” at the same time (defect), their transmissions collide, and neither succeeds.

The same logic applies to shared spectrum access. It is in neither device’s interest to dominate the channel or transmit continuously. If every device transmitted all the time, the medium would become saturated with interference, and no communication would be possible.

### 3.2.1 Single-Stage Scenario 

Imagine a banker with a chest full of gold coins inviting you and another player to take part in a simple game. Each of you has two possible actions: you can either cooperate or defect. The outcome of the game depends on what both of you choose. If you both decide to cooperate, the banker rewards each of you with three gold coins. If one of you cooperates while the other defects, the defector walks away with five coins while the cooperator gets nothing. And if both of you defect, you each receive just a single coin.

Your goal is straightforward: earn as many coins as possible. So you think through your options. Suppose your opponent decides to cooperate. You could cooperate as well and receive three coins, but if you defect instead, you would get five. In that case, defecting is clearly better. But what if your opponent defects? If you cooperate, you get nothing, whereas if you also defect, you at least receive one coin. Once again, defecting seems like the better choice. This means that no matter what your opponent does, defecting always gives you a higher payoff.

If your opponent reasons the same way, they will also choose to defect. And so, despite both of you acting perfectly rationally, you end up in a situation where you each receive only one coin, far worse than the three coins you both would have earned had you cooperated. 

This is the heart of the Prisoner’s Dilemma: individual rationality leads to a collectively inferior outcome.


<center>
<img src="../../.admin/notebooks/picture/PrisonnersDilemma.png" 
     alt="Prisoner's Dilemma" 
     width="50%"
     style="filter: grayscale(60%);">
     <p><em>Payoff matrix of Prisoner's Dilemma.</em></p>
</center>


### 3.2.2 Multi-Stage Scenario

Everything changes when the game is played repeatedly. In a multi-shot Prisoner’s Dilemma, players interact over many rounds and can react to each other’s past choices. Defecting once may bring a short-term advantage, but repeated defection often triggers retaliation, leading to several rounds of mutual defection where both players earn very little. By contrast, maintaining cooperation can yield many rounds of mutually beneficial outcomes.

This idea maps directly to wireless communication. Devices share the spectrum continuously, not just once. A device that “defects” by always transmitting aggressively pushes others to do the same, creating congestion and collisions. But when devices behave cooperatively, using fair backoff or taking turns - the overall throughput improves.

In repeated interactions, the goal is not just to win a single round, but to establish a pattern of behavior that keeps the system stable and efficient. This sets the stage for strategies like Tit-for-Tat, which we will explore next.




In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import Dropdown, IntSlider, VBox, HBox, Output
from IPython.display import display

# Payoff matrix: (A, B)
PAYOFFS = {
    ('C', 'C'): (3, 3),
    ('C', 'D'): (0, 5),
    ('D', 'C'): (5, 0),
    ('D', 'D'): (1, 1),
}

def simulate(strategy_a: str, strategy_b: str):
    move_a = 'C' if strategy_a == 'Always cooperate' else 'D'
    move_b = 'C' if strategy_b == 'Always cooperate' else 'D'
    
    a_cum_hist, b_cum_hist = [], []
    a_inst_hist, b_inst_hist = [], []
    
    a_total = b_total = 0
    rounds = 10
    
    for _ in range(rounds):
        da, db = PAYOFFS[(move_a, move_b)]
        a_total += da
        b_total += db
        
        # cumulative
        a_cum_hist.append(a_total)
        b_cum_hist.append(b_total)
        
        # instantaneous
        a_inst_hist.append(da)
        b_inst_hist.append(db)
    
    return a_cum_hist, b_cum_hist, a_inst_hist, b_inst_hist

# Widgets
strategy_options = ['Always cooperate', 'Always defect']
w_a = Dropdown(options=strategy_options, value='Always defect', description='Device A:')
w_b = Dropdown(options=strategy_options, value='Always cooperate', description='Device B:')

out = Output()

def update_plot(*_):
    with out:
        out.clear_output(wait=True)
        a_cum, b_cum, a_inst, b_inst = simulate(w_a.value, w_b.value)
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
        
        # ---- Left: cumulative points ----
        ax1.plot(a_cum, 'o-', label='Device A', linewidth=3, markersize=8)
        ax1.plot(b_cum, 's-', label='Device B', linewidth=2, markersize=6)
        ax1.set_xlabel('Round')
        ax1.set_ylabel('Cumulative Points')
        ax1.set_title('Cumulative Points over Rounds')
        ax1.set_ylim(-1, 60)
        ax1.set_xticks(range(10))
        ax1.grid()
        ax1.legend()
        
        # ---- Right: instantaneous points ----
        ax2.plot(a_inst, 'o-', label='Device A', linewidth=3, markersize=8)
        ax2.plot(b_inst, 's-', label='Device B', linewidth=2, markersize=6)
        ax2.set_xlabel('Round')
        ax2.set_ylabel('Instantaneous Points')
        ax2.set_title('Instantaneous Points per Round')
        ax2.set_ylim(-1, 6)
        ax2.set_xticks(range(10))
        ax2.grid()
        ax2.legend()
        
        plt.tight_layout()
        plt.show()

# Wire up callbacks and render
for w in (w_a, w_b):
    w.observe(update_plot, names='value')

update_plot()
display(VBox([HBox([w_a, w_b]), out]))


### 3.2.3 Tit-for-Tat and Cooperation
The surprising power of simple cooperative strategies was first demonstrated in the early 1980s by political scientist Robert Axelrod. In his famous computer tournament, described in “The Evolution of Cooperation” (Axelrod & Hamilton, 1981), researchers from diverse fields submitted algorithms to play a repeated Prisoner’s Dilemma against one another. [1] [2] [3]

Despite receiving many complex and carefully engineered strategies, the one that consistently performed best was also one of the simplest: Tit-for-Tat.


Tit-for-Tat begins by cooperating on the first move and then simply copies the opponent’s previous action. If the opponent cooperates, Tit-for-Tat cooperates in return; if the opponent defects, Tit-for-Tat immediately retaliates by defecting in the next round, but returns to cooperation as soon as the opponent does. Axelrod showed that this strategy not only scores well individually but also promotes stable cooperation across the entire population.

Its success can be explained through four essential characteristics shared by effective, balanced long-term strategies:

- Nice: It never defects first. By starting with cooperation, it encourages trust and avoids unnecessary conflict.

- Retaliatory: It responds to defection instantly, preventing exploitation and signaling that unfair behavior will not be tolerated.

- Forgiving: Once the opponent returns to cooperation, Tit-for-Tat also cooperates again, allowing relationships to recover instead of spiraling into endless retaliation.

- Clear: Its behavior is easy to understand. Opponents quickly learn that cooperation is rewarded and defection is punished.

Because of these properties, Tit-for-Tat has become a foundational model for fairness and stability in repeated interactions, not just in game theory, but in economics, biology, social science, and computer networks. In distributed systems, for example, similar principles appear in protocols that reward cooperative behavior (like backing off when the channel is busy) while discouraging aggressive or selfish strategies that would otherwise harm the whole system. [4]


In [ ]:
from ipywidgets import Button, HBox, VBox, HTML, IntSlider, Dropdown, Output
from IPython.display import display, clear_output
import random
import numpy as np
import matplotlib.pyplot as plt

class TitForTatGameUI:
    def __init__(self):
        # --- Initialize core state early ---
        self.total_rounds = 10
        self.round_index = 1
        self.your_score = 0
        self.agent_score = 0
        self.history = {'you': [], 'agent': [], 'payoffs': []}

        # --- Game parameters ---
        self.rounds_slider = IntSlider(description='Rounds', min=5, max=30, step=1, value=10, continuous_update=False)
        self.opponent_dropdown = Dropdown(description='Agent', options=[
            'Tit-for-Tat', 'Always Cooperate', 'Always Defect', 'Grudger', 'Random'
        ], value='Tit-for-Tat')

        # --- Buttons ---
        self.cooperate_btn = Button(description='🤝 Cooperate', button_style='success')
        self.defect_btn = Button(description='🚫 Defect', button_style='danger')
        self.reset_btn = Button(description='🔄 Reset')

        # --- Display elements ---
        self.status = HTML("<b>Press a button to play Round 1.</b>")
        self.scoreboard = HTML(self._scoreboard_html(0, 0, 0))
        self.log_out = Output()
        self.plot_out = Output()

        # --- Wire up events ---
        self.cooperate_btn.on_click(lambda _: self.step('C'))
        self.defect_btn.on_click(lambda _: self.step('D'))
        self.reset_btn.on_click(lambda _: self.reset_state())

        # --- Layout ---
        controls = HBox([self.rounds_slider, self.opponent_dropdown, self.reset_btn])
        actions = HBox([self.cooperate_btn, self.defect_btn])
        ui = VBox([controls, self.status, self.scoreboard, actions, self.log_out, self.plot_out])
        display(ui)

        # Start fresh
        self.reset_state()

    # --- Helpers ---
    def _payoff(self, you, agent):
        payoff = {
            ('C', 'C'): (3, 3),
            ('C', 'D'): (0, 5),
            ('D', 'C'): (5, 0),
            ('D', 'D'): (1, 1),
        }
        return payoff[(you, agent)]

    def _agent_move(self):
        strategies = {
            'Tit-for-Tat': lambda: 'C' if self.round_index == 1 else self.history['you'][-1],
            'Always Cooperate': lambda: 'C',
            'Always Defect': lambda: 'D',
            'Grudger': lambda: 'D' if 'D' in self.history['you'] else 'C',
            'Random': lambda: random.choice(['C', 'D'])
        }
        return strategies.get(self.opponent_dropdown.value, lambda: 'C')()

    def _scoreboard_html(self, round_i, you, agent):
        return f"""
        <div style="display:flex;gap:2rem;">
          <div><b>Round:</b> {round_i}/{self.total_rounds}</div>
          <div><b>Your score:</b> {you}</div>
          <div><b>Agent score:</b> {agent}</div>
        </div>
        """

    # -----------------------------------------------------------
    # Two plots + markers, also shown before first move
    # -----------------------------------------------------------
    def _render_plot(self):
        with self.plot_out:
            clear_output(wait=True)

            # Create side-by-side figure
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

            if self.history['payoffs']:
                payoffs = np.array(self.history['payoffs'])

                # --- Cumulative ---
                cum_you = np.concatenate(([0], np.cumsum(payoffs[:, 0])))
                cum_agent = np.concatenate(([0], np.cumsum(payoffs[:, 1])))

                # --- Instantaneous ---
                inst_you = payoffs[:, 0]
                inst_agent = payoffs[:, 1]

                # LEFT: cumulative
                ax1.plot(cum_you, 'o-', label='You', linewidth=3, markersize=8)
                ax1.plot(cum_agent, 's-', label='Agent', linewidth=2, markersize=6)

                # RIGHT: instantaneous
                ax2.plot(inst_you, 'o-', label='You', linewidth=3, markersize=8)
                ax2.plot(inst_agent, 's-', label='Agent', linewidth=2, markersize=6)

            else:
                # No rounds played yet -> just nice empty axes
                cum_you = np.array([0])
                cum_agent = np.array([0])
                # no lines needed; just set limits

                # LEFT: empty cumulative axes
                # (no plot() call, just axes setup)

                # RIGHT: empty instantaneous axes
                # (no plot() call, just axes setup)

                pass  # we just fall through to axis formatting below

            # --- Common formatting for cumulative (left) ---
            ax1.set_xlabel('Round')
            ax1.set_ylabel('Cumulative score')
            ax1.set_title('Cumulative Payoff')
            ax1.set_xticks(range(self.total_rounds + 1))
            ax1.set_ylim(-1, self.total_rounds * 5 + 5)
            ax1.grid()
            if self.history['payoffs']:
                ax1.legend()

            # --- Common formatting for instantaneous (right) ---
            ax2.set_xlabel('Round')
            ax2.set_ylabel('Instantaneous score')
            ax2.set_title('Round-by-Round Payoff')
            ax2.set_xticks(range(1, self.total_rounds + 1))
            ax2.set_ylim(-1, 6)
            ax2.grid()
            if self.history['payoffs']:
                ax2.legend()

            plt.tight_layout()
            plt.show()

    # --- Core logic ---
    def reset_state(self):
        self.total_rounds = int(self.rounds_slider.value)
        self.round_index = 1
        self.your_score = 0
        self.agent_score = 0
        self.history = {'you': [], 'agent': [], 'payoffs': []}
        self.status.value = "<b>New game started.</b> Choose your move for Round 1."
        self.scoreboard.value = self._scoreboard_html(0, 0, 0)

        with self.log_out:
            clear_output(wait=True)
     
        # Show empty plots immediately
        self._render_plot()

    def step(self, your_move):
        if self.round_index > self.total_rounds:
            self.status.value = "<b>Game over.</b> Press Reset to play again."
            return

        agent_move = self._agent_move()
        y, a = self._payoff(your_move, agent_move)
        
        self.your_score += y
        self.agent_score += a
        self.history['you'].append(your_move)
        self.history['agent'].append(agent_move)
        self.history['payoffs'].append((y, a))

        self.scoreboard.value = self._scoreboard_html(self.round_index, self.your_score, self.agent_score)
        self.round_index += 1

        if self.round_index > self.total_rounds:
            if self.your_score > self.agent_score:
                outcome = "You outperformed the agent!"
            elif self.your_score < self.agent_score:
                outcome = "Agent outperformed you."
            else:
                outcome = "It's a tie."

            self.status.value = f"<b>Game over.</b> {outcome} Try a different agent or press Reset."
        else:
            self.status.value = f"Choose your move for Round {self.round_index}."
        
        self._render_plot()

# Instantiate the UI
TitForTatGameUI()


### 3.3 Wireless Spectrum Sharing as a Repeated Game

Wireless devices operate in a shared medium where many independent agents compete for access to the same radio channel. When two devices transmit simultaneously, their signals interfere and neither succeeds; when all devices defer excessively, the channel remains underutilized. This tension maps naturally to the logic of the Prisoner’s Dilemma: each device would like to transmit freely as a selfish move, but if everyone behaves maximally selfishly at the same time, everyone performs poorly. Conversely, if every device is too polite, the network wastes valuable capacity. The challenge is therefore not to eliminate competition but to manage it and to arrive at behaviors that balance efficiency and fairness in a distributed way.


#### 3.3.1 The Wireless Channel as a Shared Resource

From a game-theoretic perspective, every transmission attempt is a move in a repeated interaction. Cooperative behavior corresponds to transmitting cautiously by sensing the medium, waiting, or backing off when necessary, while aggressive behavior corresponds to transmitting frequently and seizing every opportunity regardless of others. If all devices act aggressively, collisions dominate and throughput collapses, representing the wireless equivalent of mutual defection. If all devices act too conservatively, the channel remains idle, creating an over-cooperative regime with inefficient use of resources. Real networks operate between these extremes, where devices must observe the environment and gradually adapt their behavior to one another, much like players in a repeated dilemma who learn to stabilize patterns of cooperation.


#### 3.3.2 CSMA/CA: Distributed Coordination Through Simple Rules

Modern wireless systems such as IEEE 802.11 (Wi-Fi) rely on **Carrier Sense Multiple Access with Collision Avoidance (CSMA/CA)** to coordinate access to the shared medium. CSMA/CA is intentionally simple, yet it produces remarkably stable behavior across many independent devices.

At a high level, the protocol operates as follows: Devices first sense the medium and transmit only when the channel appears idle. Instead of transmitting immediately, each device performs a random backoff by drawing a waiting time from a contention window and counting it down. The counter is frozen whenever the channel becomes busy and resumes once it is free again. Transmission occurs when the counter reaches zero. If a collision happens, the device increases the size of its contention window, making its future attempts more conservative.

This last step is not “retaliation” in a game-theoretic sense - devices do not punish one another - but it is **reactive adaptation**: when the network becomes congested, devices space out their attempts to restore stability. After a successful transmission, the window resets to a smaller value, allowing the device to behave more aggressively again. CSMA/CA shares the same essential qualities as Tit-for-Tat, which promote stable cooperation in repeated interactions:

- reactivity: devices adjust behavior based on what they observe  
- predictability: simple rules that all participants understand  
- forgiveness: behavior becomes more aggressive again once conditions improve  
- symmetry: all devices follow the same rules

From these local rules, a global structure of fairness and efficient channel use emerges.

#### 3.3.3 Channel Access Simulation

To make these ideas concrete, we include a small simulation where each device has a behavior slider ranging from very passive to very aggressive. The model visualizes per-device throughput, collision rate, and overall channel utilization. By experimenting with different settings, we see that fully aggressive devices create many collisions, fully passive devices underuse the medium, and balanced behavior achieves the best overall performance. This mirrors the role of moderation and adaptation in repeated strategic interactions.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, IntSlider, FloatSlider, Button, Output, Label, Layout
from IPython.display import display, clear_output

def simulate_channel(agg_list, slots=20000):
    """
    agg_list: list of aggressiveness values in [0,1]
    slots: number of time slots
    """
    rng = np.random.default_rng()
    n = len(agg_list)
    a = np.array(agg_list, dtype=float)

    successes = np.zeros(n, dtype=int)
    collisions = 0
    idle = 0

    for _ in range(slots):
        attempts = rng.random(n) < a
        k = attempts.sum()

        if k == 0:
            idle += 1
        elif k == 1:
            idx = np.argmax(attempts)
            successes[idx] += 1
        else:
            collisions += 1

    thr = successes / slots
    util = thr.sum()
    coll_rate = collisions / slots
    idle_rate = idle / slots

    return {
        "throughput": thr,
        "utilization": util,
        "collision_rate": coll_rate,
        "idle_rate": idle_rate,
    }


def make_sliders(n):
    sliders = []
    for i in range(n):
        s = FloatSlider(
            value=0.3,
            min=0.0,
            max=1.0,
            step=0.01,
            description=f"Device {i+1}",
            continuous_update=False,
            readout_format=".2f",
            layout=Layout(width="350px")
        )
        sliders.append(s)
    return sliders


def build_ui():
    n_devices_slider = IntSlider(
        value=4, min=2, max=10, step=1,
        description="Devices", continuous_update=False
    )

    behavior_sliders = make_sliders(n_devices_slider.value)
    behavior_box = VBox(behavior_sliders)

    run_btn = Button(description="Run Simulation", button_style="primary")
    out = Output()

    def on_n_devices_change(change):
        new_n = change["new"]
        behavior_box.children = make_sliders(new_n)

    n_devices_slider.observe(on_n_devices_change, names="value")

    def run_simulation(*args):
        with out:
            clear_output(wait=True)

            agg_list = [s.value for s in behavior_box.children]
            res = simulate_channel(agg_list=agg_list)

            n = len(agg_list)

            # --- Plots ---
            fig = plt.figure(figsize=(10, 4))

            # Throughput
            ax1 = fig.add_subplot(1, 2, 1)
            idx = np.arange(n)
            ax1.bar(idx, res["throughput"])
            ax1.set_xticks(idx, [f"D{i+1}" for i in range(n)])
            ax1.set_ylabel("Throughput (successes per slot)")
            ax1.set_title("Per-Device Throughput")

            # Channel breakdown
            ax2 = fig.add_subplot(1, 2, 2)
            props = [res["utilization"], res["collision_rate"], res["idle_rate"]]
            labels = ["Success", "Collisions", "Idle"]
            ax2.bar(range(3), props)
            ax2.set_xticks(range(3), labels, rotation=15)
            ax2.set_ylim(0, 1)
            ax2.set_title("Channel Usage Breakdown")

            plt.tight_layout()
            plt.show()

            # Summary
            print("=== Simulation Summary ===")
            print(f"Devices: {n}")
            print(f"Utilization: {res['utilization']:.3f}")
            print(f"Collisions:  {res['collision_rate']:.3f}")
            print(f"Idle:        {res['idle_rate']:.3f}")
            print()
            for i, thr in enumerate(res["throughput"], 1):
                print(f"  D{i}: throughput={thr:.5f}   aggressiveness={agg_list[i-1]:.2f}")

    run_btn.on_click(run_simulation)

    ui = VBox([
        Label("Demo: Spectrum Sharing — Cooperation vs. Aggression"),
        n_devices_slider,
        Label("Device aggressiveness (0 = passive, 1 = always attacking the channel):"),
        behavior_box,
        run_btn,
        out
    ])

    return ui


ui = build_ui()
display(ui)


### 3.4 Beyond Game Theory: Control, Intelligence, and Broader Implications

#### 3.4.1 A Control-Theoretic View of Spectrum Sharing

The interaction of wireless devices on a shared channel can be understood not only as a repeated strategic game but also as a distributed control problem. Each device continually observes the state of the medium, maintains internal variables that reflect its recent experience, and adjusts its behavior in response to what it senses. A collision acts as a corrective signal pushing the device toward more cautious operation, while a successful transmission shifts it back toward a more active regime. Although the mechanisms are simple, the pattern is unmistakably that of feedback control: measure the environment, compare it with expectations, and take action to maintain stability. When devices react too aggressively, the system exhibits instability and oscillations; when they react too conservatively, the channel remains underutilized. The overall balance that emerges is the result of many small control loops operating in parallel, each independently seeking equilibrium within a shared and dynamic environment.

#### 3.4.2 From Fixed Rules to Intelligent Adaptation

In recent years, these ideas have been extended by introducing learning and adaptive intelligence into the process. Rather than relying solely on fixed backoff strategies, devices can now adjust their behavior based on observed patterns, predicted congestion, or inferred intentions of their neighbors. Reinforcement learning provides a framework in which agents gradually discover strategies that improve their performance while coexisting with others, and multi-agent learning explores how such strategies evolve when many devices adapt simultaneously. Although the underlying principles remain rooted in feedback and adjustment, the addition of learning allows the system to discover behaviors that are more nuanced than what simple deterministic rules can offer. In this sense, the boundary between control theory and artificial intelligence becomes fluid, with both contributing to more capable and robust sharing of the wireless medium.

#### 3.4.3 Patterns That Generalize Across Systems and Disciplines

The dynamics we observe in wireless networks are not unique to communication systems. Similar patterns appear in economics, where agents adjust their actions based on market conditions; in ecology, where species regulate their activity in response to competition and resource availability; and in social systems, where cooperation, restraint and adaptation shape group behavior. In each case, stable outcomes arise not through central coordination but through repeated local interactions governed by observation, moderation and feedback. The same logic applies in many aspects of daily life: success often depends on reading the situation, responding proportionally, adjusting to changing conditions and avoiding the extremes of overreaction or passivity. By studying these principles in the context of spectrum sharing, we gain insight into a broader class of systems where distributed actors must coexist, cooperate and compete within shared constraints.


### 3.5 Further Readings

[1] R. Axelrod (1984) [**The Evolution of Cooperation.**](https://doi.org/10.2307/j.ctv1xxs0fp) Basic Books.

[2] R. Axelrod (1980) [**Effective Choice in the Prisoner's Dilemma.**](https://doi.org/10.1177/002200277902400101) *Journal of Conflict Resolution*, 24(1), 3–25. 

[3] R. Axelrod and W. D. Hamilton (1981) [**The Evolution of Cooperation.**](https://doi.org/10.1126/science.7466396) *Science*, 211(4489), 1390–1396.


[4] Veritasium (2020) [**This Game Theory Problem Will Change the Way You See the World.**](https://www.youtube.com/watch?v=mScpHTIi-kM) *YouTube*, uploaded by Veritasium, 14 Mar. 2020.

----
### 3.6 Hands-On Exercises

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

**If you know that the other device will cooperate in the Prisoner’s Dilemma scenario, what is the best action for you to take?**  Answer:  

(add your answer here)


**If you know that the other device will defect, what is the best action for you to take?**  Answer:  

(add your answer here)


**In the simulation against the Tit-for-Tat agent, try to outperform it. Describe the strategy you used and explain why it works.**  Answer:  

(add your answer here)

**Interact with the Grudger agent in the simulation. Observe its behavior and explain what you believe its strategy is. When does it cooperate, and when does it retaliate?**  Answer:  

(add your answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)